# 9-1절 연습 문제 풀이

이 노트북은 9-1절 연습 문제(9-1 ~ 9-4)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch09/09-01_example.ipynb`를 참고한다.
- 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

**학습 환경에 관한 안내**: 9-1절 본문은 배치 크기 1로 학습하지만, 이 노트북은 **9-2절에서 도입하는 미니배치(배치 크기 32) 방식**으로 학습한다. 배치 크기 1로 세 번 학습하면 한 시간이 넘는데, 이 절의 연습 문제가 묻는 것(교사 강제 비율의 영향, 양방향 인코더의 효과)은 배치 구성과 무관하기 때문이다. 대신 **모든 비교를 같은 조건에서** 수행해 결론이 흔들리지 않게 했다.

## 공통 준비

In [1]:
# 환경 설정 (code_reference 모듈 임포트 경로와 시각화 설정)
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

common.set_korean_plot_env()
viz.configure(save_grayscale=False)

SEED = 42
common.set_seed(SEED, deterministic=True)
device = common.get_device()

CUDA를 사용합니다.


In [2]:
# 날짜 데이터 생성 함수 (본문 예제와 동일)
import random
import string
from datetime import datetime, timedelta

src_formats = [
    '%d %B %Y',     # 01 February 2026
    '%d %b %Y',     # 01 Feb 2026
    '%B %d, %Y',    # February 01, 2026
    '%b %d, %Y',    # Feb 01, 2026
    '%m/%d/%Y',     # 02/01/2026
    '%Y/%m/%d',     # 2026/02/01
    '%d-%m-%Y',     # 01-02-2026
    '%Y-%m-%d',     # 2026-02-01
]


def choice_random_dates(sample_size=1, start_date=None, end_date=None):
    if start_date is None:
        start_date = datetime(1900, 1, 1)
    if end_date is None:
        end_date = datetime(2050, 12, 31)
    days_between = (end_date - start_date).days
    datetime_list = []
    for _ in range(sample_size):
        days_after = random.randrange(days_between)
        datetime_list.append(start_date + timedelta(days=days_after))
    return datetime_list


def generate_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(date.strftime(src_format))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates


def add_random_noise(text, max_length=40):
    noise_chars = (
        string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
    )
    prefix, suffix = '', ''
    remaining = max_length - len(text)
    if remaining > 0:
        prefix_length = random.randint(0, remaining)
        remaining -= prefix_length
        prefix = ''.join(random.choices(noise_chars, k=prefix_length))
    if remaining > 0:
        suffix_length = random.randint(0, remaining)
        suffix = ''.join(random.choices(noise_chars, k=suffix_length))
    return prefix + text + suffix


def generate_noisy_datepairs(date_list, src_formats=src_formats):
    src_dates, tgt_dates = [], []
    for i, date in enumerate(date_list):
        src_format = src_formats[i % len(src_formats)]
        src_dates.append(add_random_noise(date.strftime(src_format)))
        tgt_dates.append(f'{date.year}-{date.month}-{date.day}')
    return src_dates, tgt_dates


common.set_seed(SEED, deterministic=True)
date_list = choice_random_dates(4000)
src_dates, tgt_dates = generate_datepairs(date_list)
print(f'{src_dates[0]!r} -> {tgt_dates[0]!r}')

'25 September 2014' -> '2014-9-25'


In [3]:
# 어휘 사전과 데이터셋 ([코드 9-1], [코드 9-12])
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SOS_TOKEN, EOS_TOKEN, PAD_TOKEN = '<sos>', '<eos>', '<pad>'
SOS_IDX, EOS_IDX, PAD_IDX = 0, 1, 2
special_tokens = {SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX, PAD_TOKEN: PAD_IDX}


class Vocab:
    def __init__(self, sequence_list, special_tokens):
        tokens = set()
        for sequence in sequence_list:
            tokens.update(sequence)
        self.vocab = {}
        self.vocab.update(special_tokens)
        idx_start = len(special_tokens)
        for i, token in enumerate(sorted(tokens)):
            self.vocab[token] = i + idx_start
        self.itos = {idx: token for token, idx in self.vocab.items()}

    def encode(self, input_sequence):
        return [self.vocab[token] for token in input_sequence]

    def decode(self, input_ids):
        return [self.itos[idx] for idx in input_ids]

    def __len__(self):
        return len(self.vocab)


class DateDataset(Dataset):
    def __init__(self, src_dates, tgt_dates, src_vocab, tgt_vocab):
        self.samples = []
        for src_date, tgt_date in zip(src_dates, tgt_dates):
            src_ids = src_vocab.encode(src_date)
            tgt_ids = [SOS_IDX] + tgt_vocab.encode(tgt_date) + [EOS_IDX]
            self.samples.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_tensors = [torch.tensor(s, dtype=torch.long) for s in src_batch]
    tgt_tensors = [torch.tensor(t, dtype=torch.long) for t in tgt_batch]
    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


def build_loaders(src_dates, tgt_dates, batch_size=32, train_size=3000):
    src_vocab = Vocab(src_dates, special_tokens)
    tgt_vocab = Vocab(tgt_dates, special_tokens)
    train_set = DateDataset(src_dates[:train_size], tgt_dates[:train_size],
                            src_vocab, tgt_vocab)
    valid_set = DateDataset(src_dates[train_size:], tgt_dates[train_size:],
                            src_vocab, tgt_vocab)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn)
    return src_vocab, tgt_vocab, train_loader, valid_loader


src_vocab, tgt_vocab, train_loader, valid_loader = build_loaders(src_dates, tgt_dates)
print(f'입력 어휘 사전 {len(src_vocab)}, 출력 어휘 사전 {len(tgt_vocab)}')

입력 어휘 사전 43, 출력 어휘 사전 14


In [4]:
# Seq2Seq 모델 ([코드 9-3] ~ [코드 9-8], [코드 9-16]의 패킹 적용)
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim, num_layers,
                 use_packing=True):
        super().__init__()
        self.use_packing = use_packing
        self.encoder_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)

    def forward(self, src, src_lengths):
        embedded = self.encoder_embedding(src)
        if self.use_packing:
            packed = pack_padded_sequence(embedded, src_lengths.cpu(),
                                          batch_first=True, enforce_sorted=False)
            _, (hidden, cell) = self.encoder_lstm(packed)
        else:
            _, (hidden, cell) = self.encoder_lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        self.decoder_lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim,
                                    num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward_step(self, token, hidden, cell, context):
        embedded = self.decoder_embedding(token)
        rnn_input = torch.cat([embedded, context], dim=-1)
        output, (hidden, cell) = self.decoder_lstm(rnn_input, (hidden, cell))
        logits = self.fc(output.squeeze(1))
        return logits, hidden, cell

    def forward(self, tgt, hidden, cell, context, forcing_ratio=0.5):
        _, target_length = tgt.shape
        all_logits = []
        token = tgt[:, 0:1]
        for i in range(target_length):
            logits, hidden, cell = self.forward_step(token, hidden, cell, context)
            all_logits.append(logits.unsqueeze(1))
            if i + 1 < target_length:
                if random.random() < forcing_ratio:
                    token = tgt[:, i + 1:i + 2]
                else:
                    token = logits.argmax(dim=-1, keepdim=True)
        return torch.cat(all_logits, dim=1)


class DateConverter(nn.Module):
    def __init__(self, encoder, decoder, forcing_ratio=0.5):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.forcing_ratio = forcing_ratio

    def forward(self, src, tgt, src_lengths):
        hidden, cell = self.encoder(src, src_lengths)
        context = hidden[-1].unsqueeze(1)
        tgt_input = tgt[:, :-1]
        return self.decoder(tgt_input, hidden, cell, context,
                            forcing_ratio=self.forcing_ratio)

In [5]:
# 학습, 검증, 예측 함수 ([코드 9-10], [코드 9-11])
import copy

import torch.optim as optim

EMBED_DIM, HIDDEN_DIM, NUM_LAYERS = 32, 32, 1
EPOCHS, PATIENCE, LR = 120, 5, 1e-3
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, token_count = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        optimizer.zero_grad()
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
    return total_loss / token_count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """토큰 단위 손실과 샘플 단위 정확도(모든 토큰이 일치할 때만 정답)를 반환

    본문 예제와 마찬가지로 검증에는 교사 강제를 적용하지 않는다.
    """
    model.eval()
    saved_ratio = model.forcing_ratio
    model.forcing_ratio = 0.0
    total_loss, token_count = 0.0, 0
    correct, total = 0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        src_lengths = (src != PAD_IDX).sum(dim=1)
        logits = model(src, tgt, src_lengths)
        labels = tgt[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        n_tokens = (labels != PAD_IDX).sum().item()
        total_loss += loss.item() * n_tokens
        token_count += n_tokens
        pred = logits.argmax(dim=-1)
        # <pad> 위치를 제외하고 모든 토큰이 일치해야 정답
        valid = labels != PAD_IDX
        match = ((pred == labels) | ~valid).all(dim=1)
        correct += match.sum().item()
        total += labels.size(0)
    model.forcing_ratio = saved_ratio
    return total_loss / token_count, correct / total * 100


def train_model(model, train_loader, valid_loader, name,
                epochs=EPOCHS, patience=PATIENCE, lr=LR, verbose_rows=8):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    log = common.EpochLogger(epochs, target_rows=verbose_rows)
    best_loss, best_epoch, best_state, counter = float('inf'), -1, None, 0
    print(f'{name} 학습')
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss, valid_acc)
        if valid_loss < best_loss:
            best_loss, best_epoch = valid_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
    log.summary(stopped='조기 종료' if counter >= patience else None)
    if best_state is not None:
        model.load_state_dict(best_state)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
    print(f'{name}: 최적 에포크 {best_epoch}, 검증 손실 {valid_loss:.4f}, '
          f'검증 정확도 {valid_acc:.2f}%')
    return {'name': name, 'best_epoch': best_epoch,
            'valid_loss': valid_loss, 'valid_acc': valid_acc}


@torch.no_grad()
def predict(model, src_text, src_vocab, tgt_vocab, max_length=12):
    model.eval()
    src_ids = src_vocab.encode(src_text)
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    src_lengths = torch.tensor([len(src_ids)])
    hidden, cell = model.encoder(src, src_lengths)
    context = hidden[-1].unsqueeze(1)
    token = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
    result = []
    for _ in range(max_length):
        logits, hidden, cell = model.decoder.forward_step(token, hidden, cell, context)
        pred = logits.argmax(dim=-1, keepdim=True)
        if pred.item() == EOS_IDX:
            break
        result.append(pred.item())
        token = pred
    return ''.join(tgt_vocab.decode(result))


def build_model(src_vocab, tgt_vocab, forcing_ratio=0.5, use_packing=True):
    common.set_seed(SEED, deterministic=True)
    encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS,
                      use_packing=use_packing)
    decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
    return DateConverter(encoder, decoder, forcing_ratio=forcing_ratio)

---

## 연습 문제 9-1

> 모델 학습 환경에 따라 결과가 다를 수 있지만, 입력 형식에 맞지 않는 `Nov,22,1901`을 입력해도 매우 높은 확률로 정답인 `1901-11-22`로 변환된다. Seq2Seq 모델이 이런 포용성을 보이는 이유를 설명해 보자.

In [6]:
# 기본 모델을 학습한 뒤 형식에 맞지 않는 입력을 넣어 본다
results = {}
model_base = build_model(src_vocab, tgt_vocab, forcing_ratio=0.5)
results['base'] = train_model(model_base, train_loader, valid_loader, '기본 모델(교사 강제 0.5)')

기본 모델(교사 강제 0.5) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.1728       1.7630        0.00%     0:02


 15/120       0.5095       0.5842       14.10%     0:22


 30/120       0.1933       0.2831       56.40%     0:44


 45/120       0.0746       0.1408       81.30%     1:06


 60/120       0.0270       0.1017       88.00%     1:27


 75/120       0.0084       0.0576       94.40%     1:49


-------------------------------------------------------
최적 82 에포크 · 검증 손실 0.0512 · 전체 학습 시간 2:06 · (조기 종료)
기본 모델(교사 강제 0.5): 최적 에포크 82, 검증 손실 0.0512, 검증 정확도 95.70%


In [7]:
# 형식에 맞는 입력과 맞지 않는 입력을 나란히 확인
cases = [
    ('Nov 22, 1901', '학습한 형식(%b %d, %Y)'),
    ('Nov,22,1901', '형식 위반(공백 없음, 쉼표 위치 다름)'),
    ('22 Nov 1901', '학습한 형식(%d %b %Y)'),
    ('Nov22 1901', '형식 위반(공백 누락)'),
    ('1901,11,22', '형식 위반(쉼표 구분 연-월-일)'),
]
print(f"{'입력':<20}{'출력':<14}{'정답 여부':<10}설명")
print('-' * 72)
for text, note in cases:
    out = predict(model_base, text, src_vocab, tgt_vocab)
    ok = '정답' if out == '1901-11-22' else '오답'
    print(f'{text:<20}{out:<14}{ok:<10}{note}')

입력                  출력            정답 여부     설명
------------------------------------------------------------------------
Nov 22, 1901        1901-11-22    정답        학습한 형식(%b %d, %Y)
Nov,22,1901         1901-11-22    정답        형식 위반(공백 없음, 쉼표 위치 다름)
22 Nov 1901         1901-11-22    정답        학습한 형식(%d %b %Y)
Nov22 1901          1901-11-21    오답        형식 위반(공백 누락)
1901,11,22          1910-11-12    오답        형식 위반(쉼표 구분 연-월-일)


### 풀이 해설 — 연습 문제 9-1

**이유는 세 가지가 겹쳐 있다.**

**1. 글자 단위 토큰이라 '형식'이라는 단위가 애초에 없다.** 모델이 보는 것은 `N`, `o`, `v`, `,`, `2`, `2`, … 라는 글자의 나열일 뿐이다. `%b %d, %Y`라는 형식이 모델 안에 규칙으로 들어 있는 것이 아니라, **'세 글자 월 이름이 나오면 그 뒤 숫자가 일(日)'** 같은 패턴을 글자 흐름에서 통계적으로 익힌 것이다. 공백이 하나 빠지거나 쉼표 위치가 달라도 나머지 글자 배열이 유지되면 같은 패턴으로 읽힌다.

**2. 인코더가 순서를 따라 읽으며 정보를 누적한다.** LSTM은 고정된 자릿수를 세는 것이 아니라 읽은 내용을 숨겨진 상태에 쌓는다. `Nov`를 읽은 시점에 '11월'이라는 정보가 이미 상태에 담기므로, 그 뒤에 공백이 있든 쉼표가 있든 그 정보는 남아 있다.

**3. 여덟 가지 형식을 함께 학습한 것이 결정적이다.** 같은 날짜가 `22 Nov 1901`, `Nov 22, 1901`, `22-11-1901`, `1901-11-22`처럼 여덟 가지로 들어오고 정답은 하나다. 모델은 **형식을 외우는 대신 형식에 상관없이 연·월·일을 찾아내는 쪽**으로 학습할 수밖에 없다. 하나의 형식만 학습했다면 이런 포용성은 나오지 않는다.

**다만 한계도 분명하다.** 위 실행 결과에서 보듯 모든 변형을 받아 주지는 않는다. 학습 데이터의 분포에서 너무 멀어지면 틀린다. **'규칙을 이해해서'가 아니라 '비슷한 패턴을 많이 봐서'**라는 점을 놓치면 안 된다.

---

## 연습 문제 9-2

> 교사 강제 비율은 학습과 모델의 성능에 어떤 영향을 미칠까? 교사 강제의 원리를 바탕으로 먼저 답을 예상해 본 뒤, 날짜 형식 변환기 모델의 교사 강제 비율(`forcing_ratio`)을 `0.0`, `0.5`, `1.0`으로 바꿔가며 학습 과정, 모델 성능, 변환 결과로 확인해 보자.

In [8]:
# 교사 강제 비율 0.0 / 0.5 / 1.0 비교 (0.5는 위에서 학습한 기본 모델)
models = {0.5: model_base}
for ratio in (0.0, 1.0):
    model = build_model(src_vocab, tgt_vocab, forcing_ratio=ratio)
    results[ratio] = train_model(model, train_loader, valid_loader,
                                 f'교사 강제 {ratio}')
    models[ratio] = model
results[0.5] = results['base']

교사 강제 0.0 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.1764       1.7571        0.00%     0:01


 15/120       0.5293       0.5390        9.50%     0:22


 30/120       0.1927       0.2327       54.80%     0:44


 45/120       0.0708       0.1130       81.00%     1:07


...

 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.0811       1.9172        0.00%     0:02


 15/120       0.3453       0.8617       24.60%     0:22


 30/120       0.1316       0.5895       51.60%     0:44


 45/120       0.0546       0.2839       79.80%     1:07


-------------------------------------------------------
최적 54 에포크 · 검증 손실 0.2056 · 전체 학습 시간 1:27 · (조기 종료)
교사 강제 1.0: 최적 에포크 54, 검증 손실 0.2056, 검증 정확도 85.40%


In [9]:
# 주의: train_model()이 기록한 검증 정확도는 모델마다 '평가 조건'이 다르다.
#   evaluate()가 model.forward()를 호출하므로 모델의 forcing_ratio가 평가에도 적용된다.
#   비율 1.0 모델은 평가에서도 정답을 받아 가장 유리하고, 0.0 모델은 가장 불리하다.
#   공정하게 견주려면 세 모델을 모두 '교사 강제 없음'으로 다시 평가해야 한다.
print('(1) 학습 중 기록한 값 - 모델마다 평가 조건이 다르다')
print(f"{'교사 강제 비율':<16}{'최적 에포크':>10}{'검증 손실':>12}{'검증 정확도':>14}")
print('-' * 54)
for ratio in (0.0, 0.5, 1.0):
    r = results[ratio]
    print(f'{ratio:<16}{r["best_epoch"]:>10}{r["valid_loss"]:>12.4f}{r["valid_acc"]:>13.2f}%')

print()
print('(2) 세 모델을 모두 교사 강제 없이 다시 평가 - 공정한 비교')
print(f"{'교사 강제 비율':<16}{'검증 손실':>12}{'검증 정확도':>14}")
print('-' * 44)
fair = {}
for ratio in (0.0, 0.5, 1.0):
    model = models[ratio]
    saved = model.forcing_ratio
    model.forcing_ratio = 0.0          # 평가 조건을 통일
    loss, acc = evaluate(model, valid_loader, criterion, device)
    model.forcing_ratio = saved
    fair[ratio] = (loss, acc)
    print(f'{ratio:<16}{loss:>12.4f}{acc:>13.2f}%')

print()
print('같은 날짜를 여덟 가지 형식으로 입력했을 때의 변환 결과')
common.set_seed(SEED, deterministic=True)
sample_date = choice_random_dates(1)[0]
answer = f'{sample_date.year}-{sample_date.month}-{sample_date.day}'
print(f'정답: {answer}')
print(f"{'입력':<24}" + ''.join(f'{f"비율 {r}":>14}' for r in (0.0, 0.5, 1.0)))
print('-' * 66)
for fmt in src_formats:
    text = sample_date.strftime(fmt)
    outs = [predict(models[r], text, src_vocab, tgt_vocab) for r in (0.0, 0.5, 1.0)]
    print(f'{text:<24}' + ''.join(f'{o:>14}' for o in outs))

(1) 학습 중 기록한 값 - 모델마다 평가 조건이 다르다
교사 강제 비율            최적 에포크       검증 손실        검증 정확도
------------------------------------------------------
0.0                     84      0.0397        95.00%
0.5                     82      0.0512        95.70%
1.0                     54      0.2056        85.40%

(2) 세 모델을 모두 교사 강제 없이 다시 평가 - 공정한 비교
교사 강제 비율               검증 손실        검증 정확도
--------------------------------------------


0.0                   0.0397        95.00%


0.5                   0.0512        95.70%


1.0                   0.2056        85.40%

같은 날짜를 여덟 가지 형식으로 입력했을 때의 변환 결과
정답: 2014-9-25
입력                              비율 0.0        비율 0.5        비율 1.0
------------------------------------------------------------------
25 September 2014            2014-9-25     2014-9-25     2014-9-25
25 Sep 2014                  2014-9-25     2014-9-25     2014-9-25
September 25, 2014           2014-9-25     2014-9-25     2014-9-25
Sep 25, 2014                 2014-9-25     2014-9-25     2014-9-25


09/25/2014                   2014-9-25     2014-9-25     2014-9-25
2014/09/25                   2014-9-25     2014-9-25     2014-9-25
25-09-2014                   2014-9-25     2014-9-25     2014-9-25
2014-09-25                   2014-9-25     2014-9-25     2014-9-25


### 풀이 해설 — 연습 문제 9-2

**먼저 원리로 예상해 보면 이렇다.**

| 비율 | 학습 과정 | 예상되는 약점 |
|---|---|---|
| `0.0` | 늘 자기가 만든 토큰을 마중물로 쓴다 | 학습 초반 오차가 눈덩이처럼 불어나 수렴이 느리다 |
| `1.0` | 늘 정답 토큰을 마중물로 쓴다 | 학습은 빠르지만 **생성할 때는 정답이 없다** |
| `0.5` | 둘을 섞는다 | 둘의 절충 |

`1.0`의 약점을 **노출 편향**(exposure bias)이라 부른다. 학습 중에는 매 시점 올바른 마중물을
받지만 생성할 때는 자기가 만든 토큰을 받는다. 한 번 틀리면 그 뒤로는 학습 중에 본 적 없는
상황에 놓인다.

**이 노트북의 `evaluate()`는 본문 예제와 마찬가지로 검증에 교사 강제를 적용하지 않는다.**
따라서 세 모델은 학습 방식만 다를 뿐 **평가 조건은 처음부터 같다.** 학습 로그의 검증 손실을
그대로 나란히 놓고 읽어도 된다.

실행 결과의 (1)을 보면 노출 편향이 곧바로 드러난다.

| 학습 시 교사 강제 비율 | 최적 에포크 | 검증 손실 | 검증 정확도 |
|---|---|---|---|
| `0.0` | 84 | 0.0397 | 95.00% |
| `0.5` | 82 | 0.0512 | 95.70% |
| `1.0` | **54** | **0.2056** | **85.40%** |

비율 `1.0` 모델만 검증 손실이 네 배 이상 높고 정확도도 10%p 낮다. 게다가 **가장 일찍 조기
종료됐다**(54 에포크). 훈련 손실은 세 모델 중 가장 빠르게 떨어지는데 검증 손실은 일찍 바닥을
치고 올라가기 때문이다. 정답을 늘 받아 온 모델이 정답 없는 상황에 가장 약하다는 것이
숫자로 확인된다.

**(2)는 세 모델을 다시 한번 교사 강제 없이 평가한 것이다.** `evaluate()`가 이미 교사 강제를
끄고 있으므로 (1)과 완전히 같은 값이 나와야 하고, 실제로 그렇다. 평가 조건이 정말 통일되어
있는지 확인하는 셈이다.

**한 가지 더 눈여겨볼 것이 있다.** 교사 강제를 켜고 평가하면 손실은 크게 달라지지만
**완전 일치 정확도는 한 자리도 바뀌지 않는다.** 자유 실행이 모든 토큰을 맞혔다면 넣어 준
토큰이 곧 정답이라 강제해도 결과가 같고, 한 토큰이라도 틀렸다면 어느 쪽이든 오답이기
때문이다. 노출 편향은 **정확도가 아니라 손실에서** 먼저 드러난다.

마지막으로 **변환 결과 표는 `predict()`로 만든 것**이라 정답의 길이조차 주지 않는다. 세 모델의
실제 생성 능력을 눈으로 견줄 수 있는 자리다.

**이 예제에서 차이가 더 크지 않은 이유**도 짚어 둘 만하다. 출력 길이가 8~10자로 짧아 오차가
누적될 구간이 길지 않다. 번역처럼 긴 출력을 다루는 모델에서는 격차가 훨씬 커진다.
[연습 문제 9-7]의 정렬 문제(출력 40~50자)로 같은 실험을 해 보면 차이가 분명해진다.


---

## 연습 문제 9-3 [도전 문제]

> `DateConverter` 클래스로 만든 모델에 입력 오류 등의 이유로 `June, 03, 1974` 대신 `june, 03, 1974`를 입력하면 예외가 발생한다. 예외가 발생하는 이유를 찾아본 뒤, 특수 토큰 `<unk>`를 사용하여 엉뚱한 변환 결과가 나오더라도 예외는 발생하지 않도록 수정해 보자. 그리고 이 방법으로 노이즈를 추가한 날짜 문자열 변환 결과도 함께 확인해 보자.

In [10]:
# 1단계 - 예외를 직접 확인한다
try:
    predict(model_base, 'june, 03, 1974', src_vocab, tgt_vocab)
except KeyError as error:
    print(f'KeyError 발생: {error}')
    print()
    print('원인: Vocab.encode()가 어휘 사전에 없는 글자를 만나면 딕셔너리 조회에 실패한다.')
    print(f"입력 어휘 사전에 'j'가 있는가: {'j' in src_vocab.vocab}")
    print(f"입력 어휘 사전에 'J'가 있는가: {'J' in src_vocab.vocab}")

KeyError 발생: 'j'

원인: Vocab.encode()가 어휘 사전에 없는 글자를 만나면 딕셔너리 조회에 실패한다.
입력 어휘 사전에 'j'가 있는가: False
입력 어휘 사전에 'J'가 있는가: True


In [11]:
# 2단계 - <unk>를 추가한 어휘 사전
UNK_TOKEN = '<unk>'
UNK_IDX = 3
special_tokens_unk = {SOS_TOKEN: SOS_IDX, EOS_TOKEN: EOS_IDX,
                      PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}


class VocabUnk(Vocab):
    """어휘 사전에 없는 토큰을 <unk>로 대체해 인코딩하는 어휘 사전"""

    def encode(self, input_sequence):
        # 없는 토큰은 KeyError 대신 <unk>의 고유 번호로 바꾼다
        return [self.vocab.get(token, UNK_IDX) for token in input_sequence]


src_vocab_unk = VocabUnk(src_dates, special_tokens_unk)
tgt_vocab_unk = VocabUnk(tgt_dates, special_tokens_unk)

print('<unk> 적용 후 인코딩 결과')
for text in ('June, 03, 1974', 'june, 03, 1974'):
    ids = src_vocab_unk.encode(text)
    print(f'  {text!r}')
    print(f'    -> {ids}')
    print(f'    -> {"".join(src_vocab_unk.decode(ids))!r}')

<unk> 적용 후 인코딩 결과
  'June, 03, 1974'
    -> [21, 41, 35, 29, 5, 4, 8, 11, 5, 4, 9, 17, 15, 12]
    -> 'June, 03, 1974'
  'june, 03, 1974'
    -> [3, 41, 35, 29, 5, 4, 8, 11, 5, 4, 9, 17, 15, 12]
    -> '<unk>une, 03, 1974'


In [12]:
# 3단계 - <unk> 어휘 사전으로 모델을 다시 학습하고 변환 결과를 확인
train_set_unk = DateDataset(src_dates[:3000], tgt_dates[:3000],
                            src_vocab_unk, tgt_vocab_unk)
valid_set_unk = DateDataset(src_dates[3000:], tgt_dates[3000:],
                            src_vocab_unk, tgt_vocab_unk)
train_loader_unk = DataLoader(train_set_unk, batch_size=32, shuffle=True,
                              collate_fn=collate_fn)
valid_loader_unk = DataLoader(valid_set_unk, batch_size=32, shuffle=False,
                              collate_fn=collate_fn)

model_unk = build_model(src_vocab_unk, tgt_vocab_unk, forcing_ratio=0.5)
results['unk'] = train_model(model_unk, train_loader_unk, valid_loader_unk,
                             '<unk> 적용 모델')

<unk> 적용 모델 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.2177       1.7603        0.00%     0:01


 15/120       0.5596       0.6087        9.40%     0:22


 30/120       0.2154       0.3065       52.60%     0:44


 45/120       0.0953       0.1795       75.80%     1:06


 60/120       0.0391       0.1250       86.00%     1:29


 75/120       0.0170       0.0930       91.00%     1:50


-------------------------------------------------------
최적 83 에포크 · 검증 손실 0.0740 · 전체 학습 시간 2:08 · (조기 종료)
<unk> 적용 모델: 최적 에포크 83, 검증 손실 0.0740, 검증 정확도 92.80%


In [13]:
# 대소문자가 어긋난 입력과 노이즈를 추가한 입력을 함께 확인
common.set_seed(SEED, deterministic=True)
noisy_list = choice_random_dates(4)
noisy_src, noisy_tgt = generate_noisy_datepairs(noisy_list)

print('대소문자가 어긋난 입력 (예외 없이 동작해야 한다)')
for text in ('June, 03, 1974', 'june, 03, 1974', 'JUNE, 03, 1974'):
    out = predict(model_unk, text, src_vocab_unk, tgt_vocab_unk)
    print(f'  {text:<20} -> {out}')

print()
print('노이즈를 추가한 입력')
for text, answer in zip(noisy_src, noisy_tgt):
    out = predict(model_unk, text, src_vocab_unk, tgt_vocab_unk)
    mark = '정답' if out == answer else '오답'
    print(f'  {text[:38]:<40} -> {out:<12} (정답 {answer}, {mark})')

대소문자가 어긋난 입력 (예외 없이 동작해야 한다)
  June, 03, 1974       -> 1974-3-6
  june, 03, 1974       -> 1974-13
  JUNE, 03, 1974       -> 197-3-4

노이즈를 추가한 입력
  vmj$W0ci25 September 2014Scr5Wt0         -> 1950-1-20-3  (정답 2014-9-25, 오답)
  ]^oLyt^jHFExdOl:hA3{Grg6&24 Dec 1919     -> 1919-12-24   (정답 1919-12-24, 정답)
  uiy4GGsx,52p#oH?4X8=*ucCxs,June 28, 19   -> 1904-1-22-21 (정답 1904-6-28, 오답)
  @sS{5mm%V%MZG?mR^[no81I1PwX,Jan 21, 20   -> 2033-11-21-2 (정답 2033-1-21, 오답)


### 풀이 해설 — 연습 문제 9-3

**예외의 원인**은 `Vocab.encode()`의 딕셔너리 조회다.

```python
return [self.vocab[token] for token in input_sequence]
```

어휘 사전은 학습 데이터에 **실제로 나타난 글자**로만 만들어진다. 날짜 형식에서 월 이름은 항상 `June`처럼 첫 글자가 대문자이므로 소문자 `j`는 사전에 없고, `self.vocab['j']`가 `KeyError`를 낸다.

**해결은 `dict.get()`으로 기본값을 주는 한 줄이다.**

```python
return [self.vocab.get(token, UNK_IDX) for token in input_sequence]
```

**여기서 중요한 것은 `<unk>`가 무엇을 해결하고 무엇을 해결하지 못하는가다.**

- 해결하는 것: **예외로 프로그램이 멈추는 일.** 모르는 글자를 만나도 `<unk>`로 바꿔 계속 진행한다.
- 해결하지 못하는 것: **변환의 정확성.** `june`의 `j`가 `<unk>`가 되면 모델은 `<unk>une`를 보게 되고, `J`를 봤을 때와 같은 판단을 하리라는 보장이 없다.

문제 지문이 "**엉뚱한 변환 결과가 나오더라도** 예외는 발생하지 않도록"이라고 못 박은 이유가 이것이다. `<unk>`는 정확도를 위한 장치가 아니라 **견고함(robustness)을 위한 장치**다.

**노이즈 입력에서도 같은 원리가 작동한다.** 노이즈에 섞인 특수문자 중 학습 데이터에 없던 것은 모두 `<unk>`가 되어 예외 없이 처리된다. 다만 노이즈 없는 데이터로 학습한 모델이므로 변환 자체는 잘 되지 않는다. 그 해결책이 9-3절의 어텐션이다.

---

## 연습 문제 9-4 [도전 문제]

> 인코더의 LSTM 계층을 양방향(`bidirectional=True`)으로 바꿔 보자. 양방향 LSTM은 출력 텐서의 마지막 차원이 두 배가 되고, 숨겨진 상태와 셀 상태도 `(2 * num_layers, B, hidden_dim)` 형태가 된다. 이 변경에 맞춰 인코더가 반환하는 `hidden`과 `cell`을 적절히 결합해 디코더 LSTM이 받을 수 있는 형태로 만들어 보자. 양방향 인코더로 학습한 모델의 노이즈 데이터셋에 대한 성능이 단방향 인코더와 비교해 어떻게 달라지는지도 함께 확인해 보자.

In [14]:
# 양방향 LSTM 인코더
class BiEncoder(nn.Module):
    def __init__(self, src_vocab_size, embed_dim, hidden_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.encoder_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                                    batch_first=True, bidirectional=True)
        # 정방향과 역방향 상태를 이어 붙인 (2 * hidden_dim)을 hidden_dim으로 되돌린다
        self.hidden_projection = nn.Linear(hidden_dim * 2, hidden_dim)
        self.cell_projection = nn.Linear(hidden_dim * 2, hidden_dim)

    def _merge(self, state, projection):
        # state: (2 * num_layers, B, hidden_dim)
        #   -> (num_layers, 2, B, hidden_dim) -> 정/역방향을 마지막 차원에서 결합
        batch_size = state.size(1)
        state = state.view(self.num_layers, 2, batch_size, self.hidden_dim)
        merged = torch.cat([state[:, 0], state[:, 1]], dim=-1)
        return torch.tanh(projection(merged))      # (num_layers, B, hidden_dim)

    def forward(self, src, src_lengths):
        embedded = self.encoder_embedding(src)
        packed = pack_padded_sequence(embedded, src_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (hidden, cell) = self.encoder_lstm(packed)
        return self._merge(hidden, self.hidden_projection), \
               self._merge(cell, self.cell_projection)


# 형태 확인
common.set_seed(SEED, deterministic=True)
bi = BiEncoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
sample_src = torch.randint(3, 20, (4, 11))
sample_len = torch.full((4,), 11)
h, c = bi(sample_src, sample_len)
print(f'양방향 인코더 반환 형태: hidden {tuple(h.shape)}, cell {tuple(c.shape)}')
print(f'디코더가 기대하는 형태:  (num_layers={NUM_LAYERS}, B=4, hidden_dim={HIDDEN_DIM})')

양방향 인코더 반환 형태: hidden (1, 4, 32), cell (1, 4, 32)
디코더가 기대하는 형태:  (num_layers=1, B=4, hidden_dim=32)


In [15]:
# 노이즈 데이터셋으로 단방향과 양방향을 비교한다
common.set_seed(SEED, deterministic=True)
noisy_date_list = choice_random_dates(4000)
src_noisy, tgt_noisy = generate_noisy_datepairs(noisy_date_list)
src_vocab_n, tgt_vocab_n, train_loader_n, valid_loader_n = build_loaders(src_noisy, tgt_noisy)
print(f'노이즈 데이터셋 예시: {src_noisy[0]!r} -> {tgt_noisy[0]!r}')
print(f'입력 어휘 사전 {len(src_vocab_n)}, 출력 어휘 사전 {len(tgt_vocab_n)}')

model_uni_n = build_model(src_vocab_n, tgt_vocab_n, forcing_ratio=0.5)
results['noisy_uni'] = train_model(model_uni_n, train_loader_n, valid_loader_n,
                                   '단방향 인코더(노이즈)')

노이즈 데이터셋 예시: 'uKq5Mc2r0I!s%)25 September 2014Z' -> '2014-9-25'
입력 어휘 사전 93, 출력 어휘 사전 14
단방향 인코더(노이즈) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.2413       1.8410        0.00%     0:02


 15/120       0.8721       0.9598        0.40%     0:24


 30/120       0.4703       0.5966       11.00%     0:48


 45/120       0.2826       0.4398       26.80%     1:12


 60/120       0.1879       0.3371       42.70%     1:36


 75/120       0.1175       0.2941       54.10%     2:00


-------------------------------------------------------
최적 81 에포크 · 검증 손실 0.2722 · 전체 학습 시간 2:17 · (조기 종료)
단방향 인코더(노이즈): 최적 에포크 81, 검증 손실 0.2722, 검증 정확도 57.50%


In [16]:
common.set_seed(SEED, deterministic=True)
bi_encoder = BiEncoder(len(src_vocab_n), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
bi_decoder = Decoder(len(tgt_vocab_n), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS)
model_bi_n = DateConverter(bi_encoder, bi_decoder, forcing_ratio=0.5)
results['noisy_bi'] = train_model(model_bi_n, train_loader_n, valid_loader_n,
                                  '양방향 인코더(노이즈)')

양방향 인코더(노이즈) 학습


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/120       2.2211       1.8726        0.00%     0:02


 15/120       0.9852       1.0405        0.00%     0:30


 30/120       0.7389       0.8431        0.50%     1:00


 45/120       0.5142       0.6832        3.00%     1:31


 60/120       0.3272       0.5499       15.30%     2:02


 75/120       0.1810       0.4683       28.70%     2:31


 90/120       0.0699       0.4170       43.60%     3:00


-------------------------------------------------------
최적 96 에포크 · 검증 손실 0.3874 · 전체 학습 시간 3:22 · (조기 종료)
양방향 인코더(노이즈): 최적 에포크 96, 검증 손실 0.3874, 검증 정확도 49.80%


In [17]:
print(f"{'인코더':<22}{'최적 에포크':>10}{'검증 손실':>12}{'검증 정확도':>14}")
print('-' * 60)
for key, label in (('noisy_uni', '단방향'), ('noisy_bi', '양방향')):
    r = results[key]
    print(f'{label:<22}{r["best_epoch"]:>10}{r["valid_loss"]:>12.4f}{r["valid_acc"]:>13.2f}%')

print()
print(f'파라미터 수  단방향 {common.count_params(model_uni_n):,} / '
      f'양방향 {common.count_params(model_bi_n):,}')

인코더                       최적 에포크       검증 손실        검증 정확도
------------------------------------------------------------
단방향                           81      0.2722        57.50%
양방향                           96      0.3874        49.80%

파라미터 수  단방향 24,878 / 양방향 37,486


### 풀이 해설 — 연습 문제 9-4

**구현의 핵심은 형태를 되돌리는 방법이다.** 양방향 LSTM은 숨겨진 상태를 `(2 * num_layers, B, hidden_dim)`으로 내놓는데, 디코더는 `(num_layers, B, hidden_dim)`을 기대한다. 두 가지를 정해야 한다.

**하나, 정방향과 역방향을 어떻게 짝지을 것인가.** 파이토치는 `(층0-정방향, 층0-역방향, 층1-정방향, 층1-역방향, …)` 순서로 쌓는다. 따라서 `view(num_layers, 2, B, hidden_dim)`으로 나누면 두 번째 차원이 방향이 된다. 단순히 `view(num_layers, -1)` 같은 식으로 접으면 층과 방향이 뒤섞인다.

**둘, 두 방향을 어떻게 합칠 것인가.** 세 가지 선택지가 있다.

| 방법 | 장점 | 단점 |
|---|---|---|
| 이어 붙인 뒤 선형 계층으로 축소 | 두 방향의 정보를 모두 살린다 | 파라미터가 늘어난다 |
| 두 방향을 더하거나 평균 | 파라미터가 늘지 않는다 | 두 방향을 같은 비중으로 섞는다 |
| 정방향만 사용 | 가장 간단하다 | 양방향으로 만든 의미가 없다 |

이 풀이는 첫 번째를 택했다. **`hidden`과 `cell`에 각각 별도의 선형 계층을 둔 것**에 주목하자. 둘은 성격이 다른 기억(단기/장기)이므로 같은 변환을 쓰면 정보가 섞인다. `tanh`를 씌운 것은 LSTM 상태의 값 범위(`-1`~`1`)에 맞추기 위해서다.

**노이즈 데이터에서 양방향이 유리하리라 기대하는 이유**는 분명하다. 노이즈가 날짜 문자열의 **앞뒤 양쪽**에 붙으므로, 단방향 인코더의 마지막 상태는 뒤쪽 노이즈를 가장 최근에 읽은 상태다. 역방향 패스가 있으면 앞쪽에서 읽어 온 정보도 함께 담긴다.

**그런데 실행 결과는 기대와 반대다.**

| 인코더 | 최적 에포크 | 검증 손실 | 검증 정확도 | 파라미터 | 학습 시간 |
|---|---|---|---|---|---|
| 단방향 | 81 | **0.2722** | **57.50%** | **24,878** | **2:17** |
| 양방향 | 96 | 0.3874 | 49.80% | 37,486 | 3:22 |

양방향이 파라미터를 1.5배 쓰고 학습도 1.5배 오래 걸리면서 정확도는 **7.7%p 낮다.** 모든 축에서 졌다.

**이유는 양방향이 병목을 넓히지 못하기 때문이다.** 역방향 패스를 더해도 **정보가 콘텍스트 벡터 하나로 압축된다는 사실은 그대로**다. 오히려 두 방향의 상태를 선형 계층으로 눌러 담는 과정이 한 번 더 들어가, 같은 크기의 벡터에 두 배의 정보를 밀어 넣는 꼴이 된다. 늘어난 파라미터만큼 학습이 어려워지는 것도 불리하게 작용한다.

**정보 병목은 인코더의 방향이 아니라 '하나의 고정 벡터'라는 구조에서 온다.** 이 결과가 그 사실을 오히려 선명하게 보여 준다. 인코더를 아무리 정교하게 만들어도 출구가 하나면 소용이 없다. 출구를 여는 것이 9-3절의 어텐션이며, 어텐션은 디코더가 인코더의 **모든 시점 출력**에 직접 접근하게 해 압축 자체를 없앤다.

한 가지 덧붙이면, 이 비교는 시드 하나에서 나온 결과다. 양방향 인코더가 원리적으로 쓸모없다는 뜻은 아니며, **어텐션과 함께 쓰면** 각 시점의 출력이 양방향 문맥을 담게 되어 실제로 도움이 된다. 여기서 드러난 것은 '병목이 있는 구조에서는 인코더 개선만으로 한계가 있다'는 점이다.
